In [1]:
from faker import Faker
import random
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from gensim.models import Word2Vec
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
import numpy as np

2024-12-21 18:56:47.112179: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-12-21 18:56:47.113003: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-12-21 18:56:47.115872: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-12-21 18:56:47.124832: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1734787607.141192  156989 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1734787607.14

In [2]:

from faker import Faker
import random
from tensorflow.keras.preprocessing.text import Tokenizer

captions_path = './train/non-radiology/captions.txt'

with open(captions_path, 'r') as file:
    captions = file.readlines()

texts = []
labels = []


In [3]:
import random

num_samples = len(captions)
texts = []
labels = []

for idx, line in enumerate(captions):
    if random.random() < 0.5:
        texts.append(line.strip())
        labels.append(1)
    else:
        texts.append(line.strip())
        labels.append(0)
    if idx % 1000 == 0:
        print(f"Processed {idx} / {num_samples} captions...")

print("Completed processing captions.")

split_index = int(0.8 * len(texts))
train_texts = texts[:split_index]
test_texts = texts[split_index:]
train_labels = labels[:split_index]
test_labels = labels[split_index:]

print(f"Number of training samples: {len(train_texts)}")
print(f"Number of testing samples: {len(test_texts)}")
print(f"Sample training text: {train_texts[0]}")
print(f"Sample label: {train_labels[0]}")


Processed 0 / 4895 captions...
Processed 1000 / 4895 captions...
Processed 2000 / 4895 captions...
Processed 3000 / 4895 captions...
Processed 4000 / 4895 captions...
Completed processing captions.
Number of training samples: 3916
Number of testing samples: 979
Sample training text: ROCO_81826	 Model bone showing the extent of graft insertion
Sample label: 1


In [4]:

# Tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts(train_texts)

# Convert text to sequences
train_sequences = tokenizer.texts_to_sequences(train_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

# Display some basic stats
print("Number of training samples:", len(train_sequences))
print("Number of test samples:", len(test_sequences))
print("Vocabulary size:", len(tokenizer.word_index) + 1)

Number of training samples: 3916
Number of test samples: 979
Vocabulary size: 18851


In [5]:
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer=Tokenizer()
tokenizer.fit_on_texts(train_texts)
train_sequences=tokenizer.texts_to_sequences(train_texts)
test_sequences=tokenizer.texts_to_sequences(test_texts)

In [6]:
word_index=tokenizer.word_index
max_sequence_length=max(len(seq) for seq in train_sequences)
from tensorflow.keras.preprocessing.sequence import pad_sequences
max_sequence_length

240

In [7]:
train_padded=pad_sequences(train_sequences,maxlen=max_sequence_length,padding="post")
test_padded=pad_sequences(test_sequences,maxlen=max_sequence_length,padding="post")
test_padded

array([[   3,  631, 6859, ...,    0,    0,    0],
       [   3,    1, 2823, ...,    0,    0,    0],
       [   3,  847,  379, ...,    0,    0,    0],
       ...,
       [   3,   41, 2247, ...,    0,    0,    0],
       [   3,  241,  154, ...,    0,    0,    0],
       [   3, 2124,    0, ...,    0,    0,    0]], dtype=int32)

In [8]:
from tensorflow.keras.utils import to_categorical
train_labels=to_categorical(train_labels)
test_labels=to_categorical(test_labels)

In [9]:
from gensim.models import Word2Vec
w2v_model=Word2Vec(sentences=[text.split() for text in train_texts],vector_size=100)

In [10]:
import numpy as np
embedding_dim=100
embedding_matrix=np.zeros((len(word_index)+1,embedding_dim))
for word,i in word_index.items():
  if word in w2v_model.wv:
    embedding_matrix[i]=w2v_model.wv[word]
embedding_matrix

array([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [-0.44885349,  0.66046077,  0.05714674, ..., -0.51049292,
        -0.14841786,  0.13488805],
       [-0.51024413,  0.70783836,  0.06932709, ..., -0.53621948,
        -0.17018716,  0.13159467],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ]])

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense

In [12]:
model=Sequential([
        Embedding(input_dim=len(word_index)+1,
                  output_dim=embedding_dim,
                  weights=[embedding_matrix],
        input_length=max_sequence_length,
        trainable=False),
        LSTM(128),
        Dense(2,activation="softmax")
])

/home/sharon/projects/all_data/env/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2024-12-21 18:59:05.514956: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [13]:
model.compile(optimizer="adam",loss="categorical_crossentropy",metrics=["accuracy"])

In [14]:
model.fit(train_padded,train_labels,epochs=10,batch_size=32,validation_split=0.2)

Epoch 1/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 11s 96ms/step - accuracy: 0.5076 - loss: 0.6935 - val_accuracy: 0.4719 - val_loss: 0.6934
Epoch 2/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 9s 88ms/step - accuracy: 0.4918 - loss: 0.6934 - val_accuracy: 0.4719 - val_loss: 0.6934
Epoch 3/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 10s 101ms/step - accuracy: 0.5184 - loss: 0.6930 - val_accuracy: 0.5293 - val_loss: 0.6918
Epoch 4/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 12s 123ms/step - accuracy: 0.5097 - loss: 0.6931 - val_accuracy: 0.4719 - val_loss: 0.6934
Epoch 5/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 12s 119ms/step - accuracy: 0.4923 - loss: 0.6933 - val_accuracy: 0.4719 - val_loss: 0.6934
Epoch 6/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 12s 121ms/step - accuracy: 0.5019 - loss: 0.7123 - val_accuracy: 0.4707 - val_loss: 0.6933
Epoch 7/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 11s 116ms/step - accuracy: 0.5004 - loss: 0.6954 - val_accuracy: 0.5293 - val_loss: 0.6996
Epoch 8/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 11s 115ms/step - accuracy: 0.4704 - loss: 0.7072 - val_accurac

#### STACKED LSTM

In [15]:
import random

num_samples = len(captions)
texts = []
labels = []

for idx, line in enumerate(captions):
    if random.random() < 0.5:
        texts.append(line.strip())
        labels.append(1)
    else:
        texts.append(line.strip())
        labels.append(0)
    if idx % 1000 == 0:
        print(f"Processed {idx} / {num_samples} captions...")

print("Completed processing captions.")

split_index = int(0.8 * len(texts))
train_texts = texts[:split_index]
test_texts = texts[split_index:]
train_labels = labels[:split_index]
test_labels = labels[split_index:]

print(f"Number of training samples: {len(train_texts)}")
print(f"Number of testing samples: {len(test_texts)}")
print(f"Sample training text: {train_texts[0]}")
print(f"Sample label: {train_labels[0]}")


Processed 0 / 4895 captions...
Processed 1000 / 4895 captions...
Processed 2000 / 4895 captions...
Processed 3000 / 4895 captions...
Processed 4000 / 4895 captions...
Completed processing captions.
Number of training samples: 3916
Number of testing samples: 979
Sample training text: ROCO_81826	 Model bone showing the extent of graft insertion
Sample label: 0


In [17]:
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer=Tokenizer()
tokenizer.fit_on_texts(train_texts)
train_sequences=tokenizer.texts_to_sequences(train_texts)
test_sequences=tokenizer.texts_to_sequences(test_texts)

In [18]:
word_index=tokenizer.word_index
max_sequence_length=max(len(seq) for seq in train_sequences)
from tensorflow.keras.preprocessing.sequence import pad_sequences
max_sequence_length
train_padded=pad_sequences(train_sequences,maxlen=max_sequence_length,padding="post")
test_padded=pad_sequences(test_sequences,maxlen=max_sequence_length,padding="post")
test_padded

array([[   3,  631, 6859, ...,    0,    0,    0],
       [   3,    1, 2823, ...,    0,    0,    0],
       [   3,  847,  379, ...,    0,    0,    0],
       ...,
       [   3,   41, 2247, ...,    0,    0,    0],
       [   3,  241,  154, ...,    0,    0,    0],
       [   3, 2124,    0, ...,    0,    0,    0]], dtype=int32)

In [19]:
from tensorflow.keras.utils import to_categorical
train_labels=to_categorical(train_labels)
test_labels=to_categorical(test_labels)
train_labels ##[cat, dog]

array([[1., 0.],
       [0., 1.],
       [0., 1.],
       ...,
       [0., 1.],
       [1., 0.],
       [0., 1.]])

In [20]:
from gensim.models import Word2Vec
w2v_model=Word2Vec(sentences=[text.split() for text in train_texts],vector_size=100)
import numpy as np
embedding_dim=100
embedding_matrix=np.zeros((len(word_index)+1,embedding_dim))
for word,i in word_index.items():
  if word in w2v_model.wv:
    embedding_matrix[i]=w2v_model.wv[word]
embedding_matrix

array([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [-0.44798079,  0.67322314,  0.05955297, ..., -0.52696878,
        -0.15210815,  0.13229381],
       [-0.51240391,  0.72506416,  0.07169823, ..., -0.53838879,
        -0.17244223,  0.12669341],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ]])

In [21]:
model=Sequential([
        Embedding(input_dim=len(word_index)+1,
                  output_dim=embedding_dim,
                  weights=[embedding_matrix],
        input_length=max_sequence_length,
        trainable=False),
        LSTM(128, return_sequences=True),  # Add return_sequences=True to the first LSTM layer
        LSTM(64),
        Dense(2,activation="softmax")
])

In [22]:
model.compile(optimizer="adam",loss="categorical_crossentropy",metrics=["accuracy"])

In [23]:
model.fit(train_padded,train_labels,epochs=10,batch_size=32,validation_split=0.2)

Epoch 1/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 17s 151ms/step - accuracy: 0.4983 - loss: 0.6937 - val_accuracy: 0.4962 - val_loss: 0.6936
Epoch 2/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 18s 185ms/step - accuracy: 0.5284 - loss: 0.6924 - val_accuracy: 0.4962 - val_loss: 0.6934
Epoch 3/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 23s 234ms/step - accuracy: 0.4915 - loss: 0.6934 - val_accuracy: 0.4949 - val_loss: 0.6937
Epoch 4/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 23s 233ms/step - accuracy: 0.5000 - loss: 0.6936 - val_accuracy: 0.4949 - val_loss: 0.6936
Epoch 5/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 27s 273ms/step - accuracy: 0.5045 - loss: 0.6931 - val_accuracy: 0.4962 - val_loss: 0.6935
Epoch 6/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 21s 214ms/step - accuracy: 0.4906 - loss: 0.6936 - val_accuracy: 0.4949 - val_loss: 0.6938
Epoch 7/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 21s 219ms/step - accuracy: 0.5014 - loss: 0.6930 - val_accuracy: 0.4962 - val_loss: 0.6932
Epoch 8/10
98/98 ━━━━━━━━━━━━━━━━━━━━ 21s 212ms/step - accuracy: 0.4962 - loss: 0.6935 - val_accu